# INVEST CREL - 합성 학습 데이터 생성기
실제 스키마(`ALTINV_CREL_TRAIN`) 기반, 부동산 금융 도메인 특성 반영

생성 후 S3 업로드 → Athena 테이블 갱신

### 1. 라이브러리 및 설정

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import awswrangler as wr
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv('.env')

np.random.seed(42)
random.seed(42)

N           = 500   # 생성 건수 (조정 가능)
S3_BUCKET   = os.getenv('S3_BUCKET', 's3-an2-mlops')
ATHENA_DB   = os.getenv('ATHENA_DB', 'mlops')
S3_OUTPUT   = os.getenv('ATHENA_S3_OUTPUT', f's3://{S3_BUCKET}/athena/')
S3_PATH     = f's3://{S3_BUCKET}/aimodel/altinv_crel_train/crel_train_sample.csv'

print(f'생성 건수: {N}')
print(f'S3 경로  : {S3_PATH}')
print(f'Athena DB: {ATHENA_DB}')


### 2. 합성 데이터 생성
> 도메인 규칙 반영: LTV↑·DSCR↓ → 부결 확률↑, 공실률↑ → 부결 확률↑

In [ ]:
SECTORS    = ['오피스', '물류', '리테일', '호텔', '주거', '복합']
IVT_TYPES  = ['선순위', '중순위', '후순위']
ASSET_DIV  = ['일반상업', '지식산업', '물류센터', '주거복합']
PROP_TYPES = ['오피스빌딩', '물류창고', '쇼핑몰', '호텔', '아파트', '오피스텔']
REGIONS    = ['국내', '해외']
CREDIT_ENH = ['있음', '없음']
ECO_GRADE  = ['G1', 'G2', 'G3', '해당없음']
ZONES      = ['서울', '경기', '인천', '부산', '대구', '기타']
DISTRICTS  = ['강남구', '서초구', '중구', '마포구', '영등포구', '기타']
AM_FIRMS   = ['미래에셋자산운용', '한국투자부동산', 'KB자산운용', '신한대체투자',
              '이지스자산운용', '코람코자산신탁', '삼성SRA자산운용', 'NH농협리츠운용',
              '하나대체투자', '마스턴투자운용']
METHODS    = ['직접', '간접']
TENANTS    = ['삼성전자', 'LG전자', 'SK하이닉스', '현대차', '롯데', 'GS리테일',
              'CJ대한통운', '쿠팡', '기타']
CREDIT_RTG = ['AA+', 'AA', 'AA-', 'A+', 'A', 'A-', 'BBB+', 'BBB']
LOCATIONS  = ['역세권 500m 이내', '대로변 접면', '산업단지 인접', '항만 인근', '고속도로IC 인접']

def rn(mu, sigma, lo, hi):
    return float(np.clip(np.random.normal(mu, sigma), lo, hi))

def rl(mu, sigma, lo, hi):
    return float(np.clip(np.random.lognormal(mu, sigma), lo, hi))

def re(scale, lo, hi):
    return float(np.clip(np.random.exponential(scale), lo, hi))

rows = []
base_date = datetime(2023, 1, 1)

for i in range(N):
    ltv         = rn(62, 10, 30, 85)
    dscr        = rn(1.3, 0.25, 0.8, 2.5)
    ln_pd       = float(np.random.choice([12,18,24,36,48,60], p=[0.05,0.15,0.35,0.25,0.15,0.05]))
    bs_itt      = rn(3.25, 0.5, 1.5, 5.0)
    etrm_rte    = re(5, 0, 40)
    mkt_etrm    = float(np.clip(etrm_rte + np.random.normal(1.5, 1), 0, 45))
    cap_rate    = rn(4.5, 0.8, 2.5, 8.0)
    cmpi_yr     = float(np.random.randint(1995, 2024))
    rmd_lsg     = re(3, 0.5, 15)
    dlb_amt     = rl(24.5, 1.2, 5e9, 5e11)
    all_pcm     = float(np.clip(dlb_amt * np.random.uniform(1.2, 2.0), 5e9, 1e12))
    trc_pi_rk   = float(np.random.choice([1,2,3], p=[0.6,0.3,0.1]))
    bdg_scl     = rl(8.5, 0.8, 500, 100000)
    nwk_scl     = rl(28, 1.5, 1e11, 1e14)
    ppo_re      = rn(45000, 15000, 10000, 150000)
    mkt_ppo_re  = float(np.clip(ppo_re * np.random.uniform(0.85, 1.15), 8000, 180000))
    mkt_dln_amt = rn(3000, 800, 500, 10000)
    te_ppo_amt  = float(np.clip(mkt_dln_amt * np.random.uniform(0.9, 1.1), 400, 12000))
    appr_amt    = float(np.clip(te_ppo_amt * np.random.uniform(0.95, 1.05), 400, 13000))
    debt_yield  = float(np.clip((ppo_re * 12 / dlb_amt) * 100, 0, 30))
    cpt_reim    = rn(30, 10, 5, 60)
    cpt_ern_rte = rn(8, 2, 3, 20)
    cpt_ern_pd  = float(np.clip(ln_pd * np.random.uniform(1.0, 1.5), 12, 120))
    rpy_rte     = float(np.random.choice([0.0, np.random.uniform(0.3, 0.8)], p=[0.7, 0.3]))

    spread  = rn(2.5, 1.0, 0.5, 6.0)
    spread += (ltv - 60) * 0.03
    spread -= (dscr - 1.3) * 0.5
    ln_itt  = float(np.clip(bs_itt + spread, 2.0, 15.0))

    score  = 0.5
    score -= (ltv - 60) * 0.015
    score += (dscr - 1.3) * 0.4
    score -= etrm_rte * 0.01
    score += rmd_lsg * 0.02
    score += (2024 - cmpi_yr) * (-0.005)
    score  = float(np.clip(score, 0.05, 0.95))
    ivt_jg = 'Y' if np.random.random() < score else 'N'

    dt      = base_date + timedelta(days=random.randint(0, 730))
    bs_ymd  = dt.strftime('%Y%m%d')
    data_ym = dt.strftime('%Y%m')
    seq     = f'JG{i+1:05d}'
    zone    = np.random.choice(ZONES)

    rows.append({
        'bs_ymd':                   bs_ymd,
        'gpt_ivt_jg_seq':           seq,
        'gpt_fl_nm':                f'투자제안_{seq}_{zone}_{np.random.choice(PROP_TYPES)}',
        'gpt_ivt_data_wrt_ym':      data_ym,
        'gpt_ivt_mth_cd':           np.random.choice(METHODS),
        'gpt_ivt_ser_dv_cd':        np.random.choice(SECTORS),
        'gpt_ivt_tp_cd':            np.random.choice(IVT_TYPES),
        'gpt_ivt_str_dv_cd':        np.random.choice(ASSET_DIV),
        'gpt_ivt_kd_cd':            np.random.choice(PROP_TYPES),
        'gpt_ivt_ara_dv_cd':        np.random.choice(REGIONS, p=[0.85,0.15]),
        'gpt_ivt_trc_pi_rk':        round(trc_pi_rk, 2),
        'gpt_ivt_dlb_rqt_amt':      round(dlb_amt, 2),
        'ln_itt':                   round(ln_itt, 4),
        'ln_pd':                    round(ln_pd, 2),
        'ltv_rte':                  round(ltv, 4),
        'gpt_ivt_cpt_ern_rte':      round(cpt_ern_rte, 4),
        'gpt_ivt_cpt_ern_pd':       round(cpt_ern_pd, 2),
        'gpt_ivt_ara_adr':          f'{zone} {np.random.choice(DISTRICTS)}',
        'gpt_ivt_dtl_lctn_txt':     np.random.choice(LOCATIONS),
        'gpt_ivt_zne_nm':           zone,
        'gpt_ivt_all_pcm_amt':      round(all_pcm, 2),
        'gpt_nwk_nm':               np.random.choice(AM_FIRMS),
        'gpt_ivt_bdg_scl_txt':      round(bdg_scl, 2),
        'gpt_ivt_nwk_ot_scl_txt':   round(nwk_scl, 2),
        'gpt_ivt_cmpi_yr':          cmpi_yr,
        'dbt_rpy_coef_rte':         round(dscr, 4),
        'gpt_ivt_rmd_lsg_ycn':      round(rmd_lsg, 2),
        'gpt_ivt_crd_rinf_txt':     np.random.choice(CREDIT_ENH, p=[0.4,0.6]),
        'gpt_ivt_etrm_rte':         round(etrm_rte, 4),
        'gpt_ivt_mkt_avg_etrm_rt':  round(mkt_etrm, 4),
        'gpt_ivt_ppo_re_amt':       round(ppo_re, 2),
        'gpt_ivt_mkt_ppo_re_amt':   round(mkt_ppo_re, 2),
        'gpt_ivt_main_hrr_txt':     np.random.choice(TENANTS),
        'gpt_ivt_hrr_ciri_txt':     np.random.choice(CREDIT_RTG),
        'gpt_ivt_mkt_avg_cpt_rte':  round(cap_rate, 4),
        'gpt_ivt_mkt_avg_dln_amt':  round(mkt_dln_amt, 2),
        'gpt_ivt_ln_pfat_txt':      round(debt_yield, 4),
        'gpt_ivt_te_ppo_amt':       round(te_ppo_amt, 2),
        'gpt_ivt_cpt_reim':         round(cpt_reim, 4),
        'gpt_ivt_appr_evl_ppo_amt': round(appr_amt, 2),
        'gpt_ivt_ecfr_gd_txt':      np.random.choice(ECO_GRADE, p=[0.1,0.2,0.2,0.5]),
        'gpt_ivt_rpy_rte':          round(rpy_rte, 4),
        'bs_itt':                   round(bs_itt, 4),
        'ivt_jg_cm_cd':             ivt_jg,
    })

df_syn = pd.DataFrame(rows)
print(f'생성 완료: {df_syn.shape}')
print('\n심사승인 분포:')
print(df_syn['ivt_jg_cm_cd'].value_counts())
print('\n수익률(ln_itt) 통계:')
print(df_syn['ln_itt'].describe())
df_syn.head(3)


### 3. S3 CSV 업로드

In [ ]:
wr.s3.to_csv(df_syn, path=S3_PATH, index=False)
print(f'S3 업로드 완료: {S3_PATH}')


### 4. Athena 테이블 갱신 (Parquet → Glue Catalog)

In [ ]:
wr.s3.to_parquet(
    df=df_syn,
    path=f's3://{S3_BUCKET}/aimodel/altinv_crel_train',  # Glue 카탈로그 등록 경로와 일치
    dataset=True,
    database=ATHENA_DB,
    table='altinv_crel_train',
    mode='overwrite',
)
print(f'Athena 테이블 갱신 완료: {ATHENA_DB}.altinv_crel_train ({len(df_syn)}건)')


### 5. 검증 - Athena 조회

In [ ]:
df_check = wr.athena.read_sql_query(
    sql='SELECT ivt_jg_cm_cd, COUNT(*) as cnt FROM mlops.altinv_crel_train GROUP BY ivt_jg_cm_cd',
    database=ATHENA_DB,
    s3_output=S3_OUTPUT
)
print('Athena 조회 결과:')
print(df_check)
